# Qwen résumé attack results: validation review

This notebook reviews the complete five-worker résumé attack run defined by `configs/qwen_attack/resume_active.json`. It works directly from shard records, so a merged result is optional.

It provides:
- a completion audit across workers and perturbation budgets;
- targeted attack success rates and perceptual metrics;
- a native-resolution viewer for clean, attacked, and diagnostic difference images; and
- exact-alignment A/B inspection without resizing or JPEG compression.

Run from the repository's `notebooks/` directory on CARC, or from a Mac checkout after copying/pulling the corresponding `outputs/qwen_attack/<run_id>/` directory.

In [1]:
from pathlib import Path
import base64
import io
import json
import sys

import ipywidgets as widgets
import matplotlib
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, clear_output, display
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'configs').exists():
    raise RuntimeError('Open this notebook from the repository root or notebooks/ directory.')

sys.path.insert(0, str(ROOT / 'src'))
from data import load_processed_rows, row_images

CONFIG_PATH = ROOT / 'configs/qwen_attack/resume_active.json'
config = json.loads(CONFIG_PATH.read_text())
RUN_ID = config['run_id']
RUN_DIR = ROOT / 'outputs/qwen_attack' / RUN_ID
RECORD_PATHS = sorted(RUN_DIR.glob('shards/worker_*/records/*.json'))
if not RECORD_PATHS:
    raise FileNotFoundError(f'No shard records found under {RUN_DIR}')

records = []
for path in RECORD_PATHS:
    record = json.loads(path.read_text())
    record['_record_path'] = str(path)
    record['_worker'] = path.parents[1].name
    records.append(record)

def first_value(record, *keys, default=None):
    for key in keys:
        value = record.get(key)
        if value is not None:
            return value
    return default

def record_id(record):
    return str(first_value(record, 'record_id', 'document_id', 'sample_id', default='unknown'))

def epsilon_of(record):
    return int(first_value(record, 'epsilon_255', 'epsilon', default=-1))

def succeeded(record):
    return bool(first_value(record, 'targeted_success_reloaded', 'saved_image_success', 'reloaded_success', 'success', default=False))

print(f'Loaded {len(records)} records from {RUN_ID}')
print(f'Run directory: {RUN_DIR}')

Loaded 296 records from qwen_resume_validation_v1
Run directory: /project2/yzhao010_1531/csci_699_new_arch/poisoned-paperwork/outputs/qwen_attack/qwen_resume_validation_v1


In [2]:
epsilons = [int(value) for value in config['attack']['epsilons_255']]
expected_documents = int(config.get('expected_validation_candidates', 0))
expected_records = expected_documents * len(epsilons)
unique_documents = sorted({record_id(record) for record in records})
complete_markers = sorted(RUN_DIR.glob('shards/worker_*/WORKER_COMPLETE.json'))
failure_paths = sorted(RUN_DIR.glob('shards/worker_*/failures/*.json'))

rows = []
for epsilon in epsilons:
    group = [record for record in records if epsilon_of(record) == epsilon]
    successes = sum(succeeded(record) for record in group)
    metric = lambda *names: [float(first_value(r, *names)) for r in group if first_value(r, *names) is not None]
    rows.append({
        'epsilon': epsilon,
        'records': len(group),
        'targeted successes': successes,
        'conditional ASR': successes / len(group) if group else np.nan,
        'median LPIPS': np.median(metric('lpips')) if metric('lpips') else np.nan,
        'median SSIM': np.median(metric('ssim')) if metric('ssim') else np.nan,
        'median runtime (s)': np.median(metric('runtime_seconds')) if metric('runtime_seconds') else np.nan,
        'median steps': np.median(metric('steps', 'completed_steps')) if metric('steps', 'completed_steps') else np.nan,
    })
summary = pd.DataFrame(rows).set_index('epsilon')
display(summary.style.format({
    'conditional ASR': '{:.1%}',
    'median LPIPS': '{:.3f}',
    'median SSIM': '{:.3f}',
    'median runtime (s)': '{:.1f}',
    'median steps': '{:.1f}',
}))

worker_counts = pd.Series([record['_worker'] for record in records]).value_counts().sort_index()
duplicate_keys = len(records) - len({(record_id(r), epsilon_of(r)) for r in records})
audit = {
    'records': f'{len(records)}/{expected_records}',
    'unique documents': f'{len(unique_documents)}/{expected_documents}',
    'targeted successes': f'{sum(succeeded(r) for r in records)}/{len(records)}',
    'worker completion markers': f'{len(complete_markers)}/5',
    'historical failure files': len(failure_paths),
    'duplicate document/epsilon keys': duplicate_keys,
}
display(pd.Series(audit, name='completion audit').to_frame())
display(worker_counts.rename('records').to_frame())

if len(records) == expected_records and len(unique_documents) == expected_documents and duplicate_keys == 0:
    display(Markdown('**Shard audit passed:** all expected experiment records are present.'))
else:
    display(Markdown('**Shard audit is incomplete:** inspect worker counts and missing records before reporting final numbers.'))

,records,targeted successes,conditional ASR,median LPIPS,median SSIM,median runtime (s),median steps
epsilon,,,,,,,
2,74,0,0.0%,0.002,0.997,242.7,100.0
4,74,1,1.4%,0.036,0.974,262.7,100.0
8,74,4,5.4%,0.191,0.907,263.0,100.0
16,74,17,23.0%,0.320,0.720,267.1,100.0


,completion audit
records,296/296
unique documents,74/74
targeted successes,22/296
worker completion markers,5/5
historical failure files,1
duplicate document/epsilon keys,0


,records
worker_00,60
worker_01,60
worker_02,60
worker_03,60
worker_04,56


**Shard audit passed:** all expected experiment records are present.

## Native-resolution visual inspection

Select a perturbation budget and outcome, then choose a résumé and page. The images are encoded losslessly as PNG and displayed at their native pixel dimensions inside horizontally scrollable panels.

- **Magnitude heatmap** uses a fixed 0-to-ε scale, making weak and concentrated changes easier to locate.
- **Absolute ×N** amplifies the RGB absolute difference for diagnosis.
- **Signed residual** maps no change to mid-gray and separates positive from negative channel changes.
- **A/B toggle** alternates the clean and attacked PNG at exactly the same coordinates; this is usually the best check for visible flicker or document changes.

In [3]:
rows_by_id = load_processed_rows(
    ROOT, split=config.get('split', 'validation'), record_ids=set(unique_documents)
)

def png_bytes(image):
    buffer = io.BytesIO()
    image.save(buffer, format='PNG', compress_level=1)
    return buffer.getvalue()

def png_data_uri(image):
    return 'data:image/png;base64,' + base64.b64encode(png_bytes(image)).decode('ascii')

def attacked_paths(record):
    image_metadata = record.get('images') or []
    raw = [item['path'] for item in image_metadata if isinstance(item, dict) and item.get('path')]
    if not raw:
        raw = first_value(record, 'attacked_image_paths', 'adversarial_image_paths', 'image_paths', default=[])
    if isinstance(raw, str):
        raw = [raw]
    found = []
    for value in raw or []:
        path = Path(value)
        if not path.is_absolute():
            candidates = [ROOT / path, RUN_DIR / path, Path(record['_record_path']).parent / path]
            path = next((candidate for candidate in candidates if candidate.exists()), candidates[0])
        found.append(path)
    if found and all(path.exists() for path in found):
        return found
    stem = Path(record['_record_path']).stem
    worker_dir = Path(record['_record_path']).parents[1]
    matches = sorted(worker_dir.glob(f'images/{stem}*.png'))
    if not matches:
        matches = sorted(worker_dir.glob(f'images/**/{stem}*.png'))
    return matches

def clean_pages(record):
    rid = record_id(record)
    if rid not in rows_by_id:
        raise KeyError(f'No processed clean row found for {rid}')
    return [image.convert('RGB') for image in row_images(rows_by_id[rid])]

def diagnostic_image(clean, attacked, kind, multiplier, epsilon):
    clean_array = np.asarray(clean, dtype=np.int16)
    attacked_array = np.asarray(attacked, dtype=np.int16)
    delta = attacked_array - clean_array
    if kind == 'Absolute ×N':
        shown = np.clip(np.abs(delta) * multiplier, 0, 255).astype(np.uint8)
        return Image.fromarray(shown), f'Absolute difference ×{multiplier}'
    if kind == 'Signed residual':
        scale = max(float(epsilon), 1.0)
        shown = np.clip(127.5 + delta * (127.5 / scale), 0, 255).astype(np.uint8)
        return Image.fromarray(shown), f'Signed residual, fixed range ±{epsilon}'
    magnitude = np.max(np.abs(delta), axis=2) / max(float(epsilon), 1.0)
    colored = matplotlib.colormaps['magma'](np.clip(magnitude, 0, 1))[:, :, :3]
    return Image.fromarray((colored * 255).astype(np.uint8)), f'Magnitude heatmap, fixed range 0–{epsilon}'

epsilon_widget = widgets.Dropdown(options=epsilons, value=max(epsilons), description='ε:')
outcome_widget = widgets.Dropdown(options=['All', 'Successful', 'Unsuccessful'], value='All', description='Outcome:')
record_widget = widgets.Dropdown(description='Résumé:', layout=widgets.Layout(width='720px'))
page_widget = widgets.Dropdown(options=[0], value=0, description='Page:')
difference_widget = widgets.Dropdown(
    options=['Magnitude heatmap', 'Absolute ×N', 'Signed residual'],
    value='Magnitude heatmap', description='Difference:'
)
multiplier_widget = widgets.IntSlider(value=16, min=1, max=64, step=1, description='Multiplier:')
render_button = widgets.Button(description='Render native resolution', button_style='primary')
viewer_output = widgets.Output()
ab_output = widgets.Output()
ab_toggle = widgets.ToggleButtons(options=[('Clean', 'clean'), ('Attacked', 'attacked')], description='A/B:')
viewer_state = {}

def record_label(index):
    record = records[index]
    mark = 'SUCCESS' if succeeded(record) else 'failure'
    prediction = first_value(record, 'prediction_normalized', 'prediction', 'output', default='?')
    return f'{record_id(record)} | {mark} | output={prediction}'

def refresh_records(*_):
    wanted = []
    for index, record in enumerate(records):
        if epsilon_of(record) != epsilon_widget.value:
            continue
        if outcome_widget.value == 'Successful' and not succeeded(record):
            continue
        if outcome_widget.value == 'Unsuccessful' and succeeded(record):
            continue
        wanted.append((record_label(index), index))
    record_widget.options = wanted
    if wanted:
        record_widget.value = wanted[0][1]

def refresh_pages(*_):
    if record_widget.value is None:
        page_widget.options = []
        return
    record = records[record_widget.value]
    count = max(len(attacked_paths(record)), int(first_value(record, 'document_page_count', 'input_page_count', 'num_pages', default=1)))
    page_widget.options = list(range(count))
    if count:
        page_widget.value = 0

def render_ab():
    if not viewer_state:
        return
    image = viewer_state[ab_toggle.value]
    with ab_output:
        clear_output(wait=True)
        display(HTML(
            '<div style="max-width:100%;overflow:auto;border:1px solid #777;background:#222;padding:8px">'
            f'<img src="{png_data_uri(image)}" width="{image.width}" height="{image.height}" '
            'style="display:block;max-width:none"></div>'
        ))

def render_selected(_=None):
    with viewer_output:
        clear_output(wait=True)
        if record_widget.value is None:
            display(Markdown('No records match this filter.'))
            return
        record = records[record_widget.value]
        page = page_widget.value
        clean = clean_pages(record)[page]
        paths = attacked_paths(record)
        if page >= len(paths):
            raise FileNotFoundError(f'Could not locate attacked page {page} for {record_id(record)}')
        attacked = Image.open(paths[page]).convert('RGB')
        if clean.size != attacked.size:
            raise ValueError(f'Dimension mismatch: clean={clean.size}, attacked={attacked.size}')
        epsilon = epsilon_of(record)
        diagnostic, diagnostic_title = diagnostic_image(
            clean, attacked, difference_widget.value, multiplier_widget.value, epsilon
        )
        prediction = first_value(record, 'prediction_normalized', 'prediction', 'output', default='?')
        clean_target = first_value(record, 'clean_target', 'clean_prediction', 'clean_label', default='?')
        target = first_value(record, 'attack_target', 'target', 'target_label', default='?')
        meta = pd.Series({
            'record': record_id(record), 'worker': record['_worker'], 'page': page,
            'epsilon': epsilon, 'targeted success': succeeded(record),
            'clean output': clean_target, 'attack target': target, 'reloaded output': prediction,
            'dimensions': f'{clean.width}×{clean.height}',
            'L∞ (255 scale)': first_value(record, 'linf_255', 'l_inf_255'),
            'LPIPS': first_value(record, 'lpips'), 'SSIM': first_value(record, 'ssim'),
        }, name='selected experiment')
        display(meta.to_frame())
        cards = []
        for title, image in [('Clean', clean), (f'Attacked ε={epsilon}', attacked), (diagnostic_title, diagnostic)]:
            cards.append(
                '<div style="display:inline-block;vertical-align:top;margin-right:16px">'
                f'<h3 style="margin:0 0 6px 0">{title}</h3>'
                f'<img src="{png_data_uri(image)}" width="{image.width}" height="{image.height}" '
                'style="display:block;max-width:none"></div>'
            )
        display(HTML(
            '<div style="width:100%;overflow:auto;white-space:nowrap;border:1px solid #777;background:#222;padding:10px">'
            + ''.join(cards) + '</div>'
        ))
        viewer_state.clear()
        viewer_state.update(clean=clean, attacked=attacked)
        display(Markdown('### Exact-alignment A/B toggle'))
        display(ab_toggle, ab_output)
        render_ab()

epsilon_widget.observe(refresh_records, names='value')
outcome_widget.observe(refresh_records, names='value')
record_widget.observe(refresh_pages, names='value')
ab_toggle.observe(lambda change: render_ab(), names='value')
render_button.on_click(render_selected)
refresh_records()
refresh_pages()
display(widgets.VBox([
    widgets.HBox([epsilon_widget, outcome_widget]),
    record_widget,
    widgets.HBox([page_widget, difference_widget, multiplier_widget]),
    render_button,
    viewer_output,
]))

## Reading the results responsibly

A record counts as a targeted success only when the saved adversarial PNG is reloaded and the model returns the exact requested target label. The amplified residual and heatmap are diagnostic displays; they are not what the model received.

For the report, pair conditional attack success rate with LPIPS, SSIM, ε, and representative native-resolution examples. A high SSIM or a visually subtle A/B comparison supports—but does not by itself prove—imperceptibility. Report the completed 296-record audit, not an interim shard summary.